# EoMT COCO → Cityscapes — fine-tuning pipeline

This notebook implements the Task 5 fine-tuning pipeline for adapting the COCO-trained EoMT checkpoint to Cityscapes semantic segmentation.

The notebook includes two branches:

- **Baseline head-only experiment**: only the `class_head` is fine-tuned using cached query-level features. This is the lightweight first experiment suggested by the project guide.
- **Gradual unfreezing pipeline**:
  - **Stage 0**: fine-tune only `class_head` + `mask_head`, keeping DINO, query embeddings and upscale blocks frozen.
  - **Stage 1**: keep DINO frozen and fine-tune the full prediction head (`q`, `class_head`, `mask_head`, `upscale`).
  - **Stage 2**: fine-tune the full prediction head plus the last DINO blocks.

The baseline and Stage 0 are independent experiments starting from the same COCO pretrained checkpoint. Stage 1 starts from Stage 0, and Stage 2 starts from Stage 1.

We do not use the backbone-token cache for the mask-based stages. That cache was unsafe with random data augmentation: tokens were cached from one augmented view, while masks could be reloaded from another randomly augmented view. This misalignment made the mask/dice loss optimize the wrong targets. For Stage 0/1/2 we therefore use the standard forward pass with live augmentation at every step.


## 1. Configuration

**This is the only section that should normally be edited**. It defines repository paths, data paths, checkpoints, configs and hyperparameters for the baseline and the three fine-tuning stages.

In [2]:
# Notebook configuration

import os
import sys
import datetime
from pathlib import Path


def log(msg: str) -> None:
    """Print a timestamped message with immediate flushing.

    This is used throughout the notebook instead of print() so that output remains
    ordered and visible in real time both on Colab and from VS Code connected to a
    Colab kernel.
    """
    print(f"[{datetime.datetime.now():%H:%M:%S}] {msg}", flush=True)


# GitHub repository
GIT_USERNAME = "ChiaraApolito"
REPO_NAME = "MaskArchitectureAnomaly_CourseProject"
BRANCH_NAME = "main"
PROJECT_FOLDER = "MaskArchitectureAnomaly_CourseProject"

# Google Drive paths
DRIVE_ROOT = Path("/content/drive/MyDrive")
LARGE_FILES = DRIVE_ROOT / "FAIML_project_and_presentation" / "01_Project" / "large_files"
PROJECT_PATH = DRIVE_ROOT / PROJECT_FOLDER          # repository cloned on Drive
PROJECT_ROOT = PROJECT_PATH / "eomt"                
REPO_URL = f"https://github.com/{GIT_USERNAME}/{REPO_NAME}.git"

# Cityscapes dataset
DRIVE_DATA_DIR = LARGE_FILES / "datasets" / "cityscapes"   # original zip files on Drive
LOCAL_DATA_DIR = Path("/content/cityscapes")               # local copy on the VM disk, faster I/O
USE_LOCAL_DATA_COPY = True
DATA_DIR = LOCAL_DATA_DIR if USE_LOCAL_DATA_COPY else DRIVE_DATA_DIR

# Pretrained weights and fine-tuning configs
COCO_WEIGHTS = LARGE_FILES / "weights" / "eomt_coco.bin"   # COCO-pretrained EoMT checkpoint
FT_CONFIG_DIR = PROJECT_ROOT / "configs" / "dinov2" / "finetuning"

BASELINE_CONFIG = FT_CONFIG_DIR / "coco_to_cityscape_freeze_all_except_class_head.yaml"
STAGE0_CONFIG = FT_CONFIG_DIR / "coco_to_cityscapes_stage0_heads.yaml"      
STAGE1_CONFIG = FT_CONFIG_DIR / "coco_to_cityscapes_freeze_backbone.yaml"   
STAGE2_CONFIG = FT_CONFIG_DIR / "coco_to_cityscapes_unfreeze_last.yaml"     


# Lightweight head-only cache
HEAD_ONLY_CACHE_DIR = LARGE_FILES / "cache" / "eomt_cache" / "cityscapes_train_head_light_cache"


# Checkpoint output directories
CKPT_ROOT = LARGE_FILES / "weights" / "finetuned"

BASELINE_CKPT_DIR = CKPT_ROOT / "coco_to_cityscapes_head_cache"
STAGE0_CKPT_DIR = CKPT_ROOT / "coco_to_cityscapes_stage0_heads"
STAGE1_CKPT_DIR = CKPT_ROOT / "coco_to_cityscapes_stage1_head"
STAGE2_CKPT_DIR = CKPT_ROOT / "coco_to_cityscapes_stage2_unfreeze_last"

# Training hyperparameters 
IMG_SIZE = [512, 512]   # 512 instead of 1024: about 1/4 of the compute and larger batches on L4

# AMP precision, torch.compile and early stopping
BASELINE_PRECISION = "16-mixed"
PRECISION = "bf16-mixed"      # bf16 is more stable than fp16 for dice/focal on L4
USE_COMPILE = True            # compile the model before fitting for faster training
EARLY_STOP_PATIENCE = 10      # epochs without val_iou_all improvement before stopping



BASELINE = {
    "config": BASELINE_CONFIG,
    "ckpt_dir": BASELINE_CKPT_DIR,
    "cache_dir": HEAD_ONLY_CACHE_DIR,
    "run_name": "coco_to_cityscapes_head_cache",
    "epochs": 50,
    "batch_size": 8,
    "precision": BASELINE_PRECISION,
    "load_class_head": False,
    "weights_name": "head_only_weights.bin",
}
STAGE0 = {
    "config": STAGE0_CONFIG,
    "ckpt_dir": STAGE0_CKPT_DIR,
    "run_name": "stage0_class_head_n_mask_head",
    "epochs": 10,
    "batch_size": 8,          
    "load_class_head": False,  
}
STAGE1 = {
    "config": STAGE1_CONFIG,
    "ckpt_dir": STAGE1_CKPT_DIR,
    "run_name": "stage1_full_head_frozen_dino",
    "epochs": 15,
    "batch_size": 8,           
    "load_class_head": True,   # starts from the class+mask heads trained in Stage 0
}
STAGE2 = {
    "config": STAGE2_CONFIG,
    "ckpt_dir": STAGE2_CKPT_DIR,
    "run_name": "stage2_full_head_n_last2_dino",
    "epochs": 40,
    "batch_size": 8,           
    "load_class_head": True,   # starts from the class+mask heads trained in Stage 1
}

log("Configuration loaded.")
log(f"  PROJECT_ROOT = {PROJECT_ROOT}")
log(f"  DATA_DIR     = {DATA_DIR}  (copia locale={USE_LOCAL_DATA_COPY})")
log(f"  IMG_SIZE     = {IMG_SIZE}")
log(f"  BASELINE: {BASELINE['epochs']} ep, batch {BASELINE['batch_size']}  |  "
    f"STAGE0: {STAGE0['epochs']} ep, batch {STAGE0['batch_size']}  |  "
    f"STAGE1: {STAGE1['epochs']} ep, batch {STAGE1['batch_size']}  |  "
    f"STAGE2: {STAGE2['epochs']} ep, batch {STAGE2['batch_size']}")

[08:16:12] Configuration loaded.
[08:16:12]   PROJECT_ROOT = /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject/eomt
[08:16:12]   DATA_DIR     = /content/cityscapes  (copia locale=True)
[08:16:12]   IMG_SIZE     = [512, 512]
[08:16:12]   BASELINE: 50 ep, batch 8  |  STAGE0: 10 ep, batch 8  |  STAGE1: 15 ep, batch 8  |  STAGE2: 40 ep, batch 8


## 2. Environment and repository setup

This section mounts Google Drive, moves to the repository root, creates the output directories, checks the required inputs and prepares the environment. \
The repository and dependencies are assumed to be handled by `00_setup.ipynb`.

In [3]:
import sys
from google.colab import drive

drive.mount("/content/drive")   

assert PROJECT_ROOT.exists(), f"PROJECT_ROOT not found: {PROJECT_ROOT} — did you run 00_setup?"
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
log(f"PWD = {Path.cwd()}")

# Create output directories and check required inputs.
BASELINE_CKPT_DIR.mkdir(parents=True, exist_ok=True)
STAGE0_CKPT_DIR.mkdir(parents=True, exist_ok=True)
STAGE1_CKPT_DIR.mkdir(parents=True, exist_ok=True)
STAGE2_CKPT_DIR.mkdir(parents=True, exist_ok=True)
assert COCO_WEIGHTS.exists(), f"Missing COCO weights: {COCO_WEIGHTS}"
assert BASELINE_CONFIG.exists(), f"Missing baseline config: {BASELINE_CONFIG}"
assert STAGE0_CONFIG.exists() and STAGE1_CONFIG.exists() and STAGE2_CONFIG.exists(), "Missing fine-tuning configs."
log("Setup completed.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[08:16:16] PWD = /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject/eomt
[08:16:19] Setup completed.


## 3. Local Cityscapes dataset copy

Copy the Cityscapes zip files to `/content`, the VM local disk, to reduce I/O latency. This is useful both on Colab and when using VS Code with a Colab kernel.

In [4]:
import shutil

if USE_LOCAL_DATA_COPY:
    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    for name in ["leftImg8bit_trainvaltest.zip", "gtFine_trainvaltest.zip"]:
        src, dst = DRIVE_DATA_DIR / name, LOCAL_DATA_DIR / name
        assert src.exists(), f"Missing file on Drive: {src}"
        if dst.exists() and dst.stat().st_size == src.stat().st_size:
            log(f"Already copied: {dst.name} ({dst.stat().st_size/1e9:.2f} GB)")
        else:
            log(f"Copying {src.name} -> {dst} ...")
            shutil.copy2(src, dst)
            log(f"OK: {dst.name} ({dst.stat().st_size/1e9:.2f} GB)")
else:
    log("Using the dataset directly from Drive.")

for name in ["leftImg8bit_trainvaltest.zip", "gtFine_trainvaltest.zip"]:
    assert (DATA_DIR / name).exists(), f"Missing zip file in DATA_DIR: {name}"
log("Dataset ready.")

[08:16:20] Copying leftImg8bit_trainvaltest.zip -> /content/cityscapes/leftImg8bit_trainvaltest.zip ...
[08:21:20] OK: leftImg8bit_trainvaltest.zip (11.59 GB)
[08:21:20] Copying gtFine_trainvaltest.zip -> /content/cityscapes/gtFine_trainvaltest.zip ...
[08:21:28] OK: gtFine_trainvaltest.zip (0.25 GB)
[08:21:28] Dataset ready.


## 4. Login WandB

In [5]:
"""WandB login without hard-coded keys.

Set the key in Colab Secrets using the name WANDB_API_KEY. This cell reads it from
there, or asks for it interactively with getpass as a fallback.
"""
os.environ["WANDB_API_KEY"] ="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"

## 5. Training utilities

This section defines the utility functions used by all fine-tuning experiments.

The notebook contains two types of training pipelines:

1. **Cached head-only baseline**  
   The COCO model is frozen, query-level features are precomputed once, and only the `class_head` is trained from cached features.

2. **No-cache gradual unfreezing stages**  
   Stage 0, Stage 1 and Stage 2 use the standard Cityscapes dataloader and run a normal forward pass. The trainable parameters are controlled by the corresponding YAML config.

The helper functions below are organized so that the baseline and the three no-cache stages share the same model-building, checkpointing, logging and export logic.


In [6]:
# Training utilities for Task 5 fine-tuning experiments.

import re
import torch
import wandb
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor, EarlyStopping

try:
    from main import LightningCLI   
except ImportError:
    from lightning.pytorch.cli import LightningCLI

DEVICE_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
log(f"Device: {DEVICE_NAME}")
if torch.cuda.is_available():
    log(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


def build_cli(config_path, ckpt_dir, run_name, max_epochs, batch_size,
              ckpt_path, load_ckpt_class_head, limit_val_batches=1.0, precision = None):
    """
    Build model, datamodule and trainer from a YAML config using LightningCLI(run=False).

    The function is shared by the cached head-only baseline and by the no-cache
    gradual unfreezing stages. The optional `precision` argument allows the baseline
    to use a different AMP precision without changing Stage 0/1/2.
    """
    if wandb.run is not None:
        wandb.finish()   # close the previous stage run: one WandB run per stage

    args = [
        "-c", str(config_path),
        "--trainer.accelerator", "gpu",
        "--trainer.devices", "1",
        "--trainer.max_epochs", str(max_epochs),
        "--trainer.precision", precision or PRECISION,                  
        "--trainer.gradient_clip_val", "0.01",            
        "--trainer.gradient_clip_algorithm", "norm",
        "--trainer.log_every_n_steps", "10",
        "--trainer.check_val_every_n_epoch", "1",
        "--trainer.limit_val_batches", str(limit_val_batches),
        "--trainer.num_sanity_val_steps", "0",
        "--trainer.default_root_dir", str(ckpt_dir),
        "--trainer.logger.init_args.project", "eomt",
        "--trainer.logger.init_args.name", run_name,
        "--trainer.logger.init_args.resume", "allow",
        "--data.path", str(DATA_DIR),
        "--data.batch_size", str(batch_size),
        "--data.img_size", str(IMG_SIZE),
        "--model.init_args.ckpt_path", str(ckpt_path),
        "--model.init_args.load_ckpt_class_head", str(load_ckpt_class_head).lower(),
    ]
    orig = sys.argv
    sys.argv = [sys.argv[0]]
    try:
        cli = LightningCLI(args=args, run=False, save_config_callback=None)
    finally:
        sys.argv = orig
    log(f"CLI built | run='{run_name}' | epochs={max_epochs} | batch={batch_size}")
    return cli


def _group(name):
    """
    Group parameter names by module.

    Transformer block parameters are grouped at block level, so the summary stays
    readable even when several layers are unfrozen.
    """
    m = re.search(r"(.*blocks\.\d+)", name)
    return m.group(1) if m else name.rsplit(".", 1)[0]


def print_model_summary(model):
    """
    Print a compact summary of the model setup and trainable parameter groups.
    """
    net = model.network
    log(f"img_size={model.img_size} | num_classes={model.num_classes} | "
        f"masked_attn={net.masked_attn_enabled} | num_q={net.num_q} | num_blocks={net.num_blocks}")
    trainable, total, groups = 0, 0, {}
    for n, p in model.named_parameters():
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()
            groups[_group(n)] = groups.get(_group(n), 0) + p.numel()
    log(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
    for k in sorted(groups):
        log(f"  [train] {k}  ({groups[k]:,})")


def _img_size_tuple(model):
    """
    Return model.img_size as a tuple.
    """
    s = model.img_size
    return (s, s) if isinstance(s, int) else tuple(s)


def sanity_check_data(cli):
    """
    Check the Cityscapes dataloading pipeline before training.

    This verifies that:
    - train and validation loaders are available;
    - image tensors have the expected shape and value range;
    - masks and labels are aligned;
    - the dataloader image size matches the model image size.
    """
    dm = cli.datamodule
    dm.setup("fit")
    train_loader, val_loader = dm.train_dataloader(), dm.val_dataloader()
    log(f"train: {len(train_loader.dataset)} img / {len(train_loader)} batch | "
        f"val: {len(val_loader.dataset)} img | batch_size={val_loader.batch_size}")
    imgs, targets = next(iter(train_loader))
    log(f"batch imgs: shape={tuple(imgs.shape)} dtype={imgs.dtype} "
        f"range=[{imgs.min():.0f},{imgs.max():.0f}]")
    t0 = targets[0]
    log(f"target[0]: keys={list(t0.keys())} | n_oggetti={t0['masks'].shape[0]} | "
        f"masks={tuple(t0['masks'].shape)} {t0['masks'].dtype} | labels={t0['labels'].tolist()}")
    assert tuple(imgs.shape[-2:]) == _img_size_tuple(cli.model), "data img_size does not match model img_size!"
    assert t0["masks"].shape[0] == t0["labels"].shape[0] > 0, "empty or misaligned masks/labels!"
    log("OK data sanity check: image and masks come from the same augmentation.")


def run_stage(cli, ckpt_dir, train_dataloaders=None, val_dataloaders=None):
    """
    Run training and return the best checkpoint path.

    If `train_dataloaders` is not provided, the function uses the standard
    datamodule. This is the path used by Stage 0, Stage 1 and Stage 2.

    If custom dataloaders are provided, they are used instead. This is needed
    only for the cached head-only baseline.
    """
    trainer = cli.trainer
    trainer.callbacks = [cb for cb in trainer.callbacks
                         if not isinstance(cb, (ModelCheckpoint, LearningRateMonitor, EarlyStopping))]
    ckpt_cb = ModelCheckpoint(
        dirpath=str(ckpt_dir), filename="best",
        monitor="metrics/val_iou_all", mode="max",
        save_top_k=1, save_last=True, auto_insert_metric_name=False,
    )

    early_cb = EarlyStopping(monitor="metrics/val_iou_all", mode="max",
                             patience=EARLY_STOP_PATIENCE)
    trainer.callbacks += [ckpt_cb, LearningRateMonitor(logging_interval="epoch"), early_cb]

    log("Training started...")

    model = torch.compile(cli.model) if USE_COMPILE else cli.model   # compile for faster training
    if train_dataloaders is None:
        # Standard case: Stage 0, Stage 1, Stage 2
        trainer.fit(model, datamodule=cli.datamodule)    # standard dataloader with live augmentation
    else:
        # Baseline head-only case: custom cached dataloader
        trainer.fit(
            model,
            train_dataloaders=train_dataloaders,
            val_dataloaders=val_dataloaders,
        )

    best_iou = float(ckpt_cb.best_model_score) * 100 if ckpt_cb.best_model_score is not None else float("nan")

    log(f"Training finished. best mIoU={best_iou:.2f} | ckpt={ckpt_cb.best_model_path}")

    return ckpt_cb.best_model_path


def export_weights(ckpt_path, out_bin):
    """
    Export a clean .bin state_dict from a Lightning checkpoint.

    The exported file can be loaded later with torch.load(..., weights_only=True).
    """
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dict = ckpt.get("state_dict", ckpt)
    state_dict = {k: v for k, v in state_dict.items() if "criterion.empty_weight" not in k}
    torch.save(state_dict, out_bin)
    log(f"Weights exported to: {out_bin}")
    return out_bin




def fit_baseline(baseline, init_weights):
    """
    Head-only baseline with cached query features.

    Pipeline:
    1. build the COCO-initialized model from the baseline config;
    2. precompute cached query features if the cache does not already exist;
    3. train only the class_head using CachedHeadDataset;
    4. validate on Cityscapes validation;
    5. return the CLI object and the best checkpoint path.
    """

    from torch.utils.data import DataLoader
    from training.head_cache_utils import (
        precompute_head_cache,
        CachedHeadDataset,
        cached_head_collate_fn,
    )

    cli = build_cli(
        config_path=baseline["config"],
        ckpt_dir=baseline["ckpt_dir"],
        run_name=baseline["run_name"],
        max_epochs=baseline["epochs"],
        batch_size=baseline["batch_size"],
        ckpt_path=init_weights,
        load_ckpt_class_head=baseline["load_class_head"],
        precision=baseline.get("precision", PRECISION),
    )

    print_model_summary(cli.model)
    sanity_check_data(cli)

    # Standard Cityscapes dataloader used only to create the cache.
    train_loader = cli.datamodule.train_dataloader()

    precompute_head_cache(
        model=cli.model,
        train_dataloader=train_loader,
        cache_dir=baseline["cache_dir"],
        overwrite=False,
    )

    cached_train_dataset = CachedHeadDataset(baseline["cache_dir"])

    cached_train_loader = DataLoader(
        cached_train_dataset,
        batch_size=baseline["batch_size"],
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        collate_fn=cached_head_collate_fn,
    )

    val_loader = cli.datamodule.val_dataloader()

    best_ckpt = run_stage(
        cli,
        baseline["ckpt_dir"],
        train_dataloaders=cached_train_loader,
        val_dataloaders=val_loader,
    )

    return cli, best_ckpt



def fit_stage(stage, init_weights):
    """
    Run one no-cache fine-tuning stage.

    This function is used by Stage 0, Stage 1 and Stage 2. It preserves the
    original behavior: standard Cityscapes dataloader, live augmentations and
    training controlled only by the YAML config.
    """
    cli = build_cli(
        config_path=stage["config"], ckpt_dir=stage["ckpt_dir"],
        run_name=stage["run_name"], max_epochs=stage["epochs"],
        batch_size=stage["batch_size"], ckpt_path=init_weights,
        load_ckpt_class_head=stage["load_class_head"],
    )
    print_model_summary(cli.model)
    sanity_check_data(cli)
    best = run_stage(cli, stage["ckpt_dir"])
    return cli, best

[08:21:45] Device: NVIDIA L4
[08:21:45] Total VRAM: 23.7 GB


In [7]:
stage1_bin = STAGE1_CKPT_DIR / "stage1_weights.bin"
assert stage1_bin.exists(), f"Missing Stage 1 weights: {stage1_bin}"

def resume_stage(stage, last_ckpt):
    """
    Resume an interrupted/unfinished stage from a Lightning checkpoint.
    This restores model weights, optimizer, scheduler and epoch counter.
    """
    cli = build_cli(
        config_path=stage["config"],
        ckpt_dir=stage["ckpt_dir"],
        run_name=stage["run_name"] + "_resume",
        max_epochs=stage["epochs"],
        batch_size=stage["batch_size"],#n_images,   #,
        ckpt_path=stage1_bin,  # serve solo per costruire il modello; il resume userà last_ckpt
        load_ckpt_class_head=stage["load_class_head"],
    )
    
    #cli.trainer.limit_train_batches = 2                   
    #cli.trainer.limit_val_batches = 2

    print_model_summary(cli.model)
    sanity_check_data(cli)

    trainer = cli.trainer
    trainer.callbacks = [
        cb for cb in trainer.callbacks
        if not isinstance(cb, (ModelCheckpoint, LearningRateMonitor, EarlyStopping))
    ]

    ckpt_cb = ModelCheckpoint(
        dirpath=str(stage["ckpt_dir"]),
        filename="best",
        monitor="metrics/val_iou_all",
        mode="max",
        save_top_k=1,
        save_last=True,
        auto_insert_metric_name=False,
    )

    early_cb = EarlyStopping(
        monitor="metrics/val_iou_all",
        mode="max",
        patience=EARLY_STOP_PATIENCE,
    )

    trainer.callbacks += [
        ckpt_cb,
        LearningRateMonitor(logging_interval="epoch"),
        #early_cb,
    ]

    model = torch.compile(cli.model) if USE_COMPILE else cli.model
    #model = cli.model
    trainer.fit(
        model,
        datamodule=cli.datamodule,
        ckpt_path=str(last_ckpt),
    )

    best = ckpt_cb.best_model_path or str(last_ckpt)
    return cli, best

## 6. Baseline — head-only fine-tuning with cached features

Before the gradual unfreezing stages, we include the lightweight head-only experiment used as the first COCO → Cityscapes adaptation baseline.

In this experiment, the COCO-trained EoMT checkpoint is adapted to Cityscapes by training only the `class_head`. To reduce training time, the frozen COCO model is first used to precompute query-level features. The cache stores only `q_features` and the corresponding matched `query_class_targets`; full masks and full targets are not stored. During training, the cached `q_features` are passed directly to the `class_head`, and the optimization uses a class-only cross-entropy loss.

This cell can be executed to reproduce the previously run baseline. If the cache already exists, `precompute_head_cache` reuses it and skips the precomputation step.

In [8]:
# # Baseline: head-only fine-tuning with cached query features.

# cli_base, best_base_ckpt = fit_baseline(
#     BASELINE,
#     init_weights=COCO_WEIGHTS,
# )

# baseline_bin = export_weights(
#     best_base_ckpt,
#     BASELINE["ckpt_dir"] / BASELINE["weights_name"],
# )

# log("Baseline head-only finished.")
# log(f"Best baseline checkpoint: {best_base_ckpt}")
# log(f"Exported baseline weights: {baseline_bin}")

## 7. Stage 0 — `class_head` + `mask_head` only

This is the most conservative gradual-unfreezing step. **Only** `class_head` and `mask_head` are trained, while DINO, query embeddings and upscale blocks remain frozen at the COCO pretrained weights. The `class_head` is randomly initialized (`load_class_head=False`, because Cityscapes has 19 classes and COCO has a different class space), while the COCO `mask_head` is allowed to adapt the masks over fixed COCO features.

Output: `stage0_weights.bin`, used as the initialization for Stage 1.

In [9]:
# # Stage 0: train only class_head + mask_head. DINO, query embeddings and upscale are frozen.
# # This stage starts from the COCO pretrained weights.
# cli0, best0_ckpt = fit_stage(STAGE0, init_weights=COCO_WEIGHTS)

# # Clean weights used to initialize Stage 1.
# stage0_bin = export_weights(best0_ckpt, STAGE0_CKPT_DIR / "stage0_weights.bin")

## 8. Stage 1 — full prediction head, frozen DINO

This stage starts from the **Stage 0** weights (`load_class_head=True`, since the head is already adapted to 19 Cityscapes classes). DINO remains frozen, while the full prediction head is unfrozen: `q`, `mask_head`, `class_head` and `upscale` (`freeze_encoder=True`). This adapts the query embeddings and the convolutional decoder in addition to classification, without modifying the pretrained backbone.

In [10]:
# # Stage 1: frozen DINO, full prediction head trainable. Starts from Stage 0 weights.
# cli1, best1_ckpt = fit_stage(STAGE1, init_weights=stage0_bin)

# # Clean weights used to initialize Stage 2.
# stage1_bin = export_weights(best1_ckpt, STAGE1_CKPT_DIR / "stage1_weights.bin")

## 9. Stage 2 — unfreeze the last 2 DINO blocks

This stage starts from the Stage 1 weights (`load_class_head=True`, since the head is already adapted to 19 Cityscapes classes) with a fresh optimizer/scheduler, not a resume. This keeps the polynomial schedule total steps consistent. The last 2 DINO blocks and the full prediction head (`q`, `mask_head`, `class_head`, `upscale`) are trainable. A low learning rate and LLRD are used to avoid disrupting the DINOv2 features.

In [11]:
# import torch
# STAGE2_LAST_CKPT = STAGE2_CKPT_DIR / "last-v1.ckpt"
# ck = torch.load(STAGE2_LAST_CKPT, map_location="cpu", weights_only=False)
# print("epoch:       ", ck.get("epoch"))         # epoche già completate
# print("global_step: ", ck.get("global_step"))
# print("max_epochs:  ", STAGE2["epochs"])


In [12]:
# import torch
# ck = torch.load(STAGE2_LAST_CKPT, map_location="cpu", weights_only=False)
# print("epoch:", ck.get("epoch"), "| max_epochs:", STAGE2["epochs"])
# for key, st in ck.get("callbacks", {}).items():
#     if "EarlyStopping" in key:
#         print(key)
#         print("  wait_count :", st.get("wait_count"))
#         print("  patience   :", st.get("patience"))
#         print("  best_score :", st.get("best_score"))
#         print("  stopped_epoch:", st.get("stopped_epoch"))

In [13]:
# STAGE2_LAST_CKPT = STAGE2_CKPT_DIR / "last-v1.ckpt"

# assert STAGE2_LAST_CKPT.exists(), f"Missing checkpoint: {STAGE2_LAST_CKPT}"

# cli2, best2_ckpt = resume_stage(STAGE2, STAGE2_LAST_CKPT)

# print("This is the best checkpoint", best2_ckpt)

# stage2_bin = export_weights(
#     best2_ckpt,
#     STAGE2_CKPT_DIR / "stage2_weights_40epochs.bin",
# )

In [14]:
# # Stage 2: unfreeze the last DINO blocks. Starts from clean Stage 1 weights.
# cli2, best2_ckpt = fit_stage(STAGE2, init_weights=stage1_bin)
# log(f"Final fine-tuned checkpoint: {best2_ckpt}")

# stage2_bin = export_weights(best2_ckpt, STAGE2_CKPT_DIR / "stage2_weights_rerun.bin")

## 10. Evaluation on Cityscapes validation

This section evaluates the four fine-tuned checkpoints on the Cityscapes validation set using the same semantic segmentation protocol adopted in Task 4.

All four fine-tuned models already predict directly in the Cityscapes label space, so no COCO → Cityscapes mapping is required. However, to keep the results comparable with the Task 4 protocol, the classes `pole`, `traffic sign` and `rider` are excluded from the mIoU computation. The reported mIoU is therefore computed on the same 16 evaluated Cityscapes classes.

The evaluated checkpoints are:

1. cached head-only baseline;
2. Stage 0: `class_head + mask_head`;
3. Stage 1: full prediction head with frozen backbone;
4. Stage 2: full prediction head plus the last DINO blocks.

In [15]:
# import os, sys
# from pathlib import Path

# print("cwd:", os.getcwd())
# print("first sys.path entries:")
# for p in sys.path[:5]:
#     print(" ", p)
# print("eval utils exists:", Path("eval/cityscapes_eval_utils.py").exists())

In [26]:
# Shared Task 5 evaluation setup

import gc
import sys
import pandas as pd
import torch

# Ensure that the local project packages are imported.
sys.path.insert(0, str(PROJECT_ROOT))

from eval.cityscapes_eval_utils import (
    CS_NAMES,
    DEFAULT_TASK4_EXCLUDED_CLASSES,
    build_eomt_eval_model,
    build_cityscapes_val_dataset,
    evaluate_cityscapes_semantic,
    make_iou_tables,
)

TASK5_EVAL_DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

# Same 16-class protocol used in Task 4:
# 5 = pole, 7 = traffic sign, 12 = rider
TASK5_EXCLUDED_CLASSES = []#DEFAULT_TASK4_EXCLUDED_CLASSES

RESULTS_ROOT = DRIVE_ROOT / "FAIML_project_and_presentation" / "01_Project" / "results"
EVAL_RESULTS_DIR = RESULTS_ROOT / "task5"
EVAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

eval_dataset = build_cityscapes_val_dataset(
    data_cls="datasets.cityscapes_semantic.CityscapesSemantic",
    data_path=DATA_DIR,
    img_size=IMG_SIZE,
    batch_size=1,
    num_workers=0,
    check_empty_targets=False,
)

print("Evaluation device:", TASK5_EVAL_DEVICE)
print("Evaluation results directory:", EVAL_RESULTS_DIR)
print("Validation images:", len(eval_dataset))
print("Evaluation img_size:", IMG_SIZE)
print("Excluded classes:", [CS_NAMES[c] for c in TASK5_EXCLUDED_CLASSES])

Evaluation device: cuda:0
Evaluation results directory: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task5
Validation images: 500
Evaluation img_size: [512, 512]
Excluded classes: []


In [27]:
# Fine-tuned checkpoints to evaluate

TASK5_MODELS = {
    # "baseline_head_only": {
    #     "config": BASELINE["config"],
    #     "weights": BASELINE["ckpt_dir"] / BASELINE["weights_name"],
    #     "description": "Cached head-only baseline",
    # },
    # "stage0_class_mask_heads": {
    #     "config": STAGE0["config"],
    #     "weights": STAGE0["ckpt_dir"] / "stage0_weights.bin",
    #     "description": "Stage 0: class_head + mask_head",
    # },
    # "stage1_full_head": {
    #     "config": STAGE1["config"],
    #     "weights": STAGE1["ckpt_dir"] / "stage1_weights.bin",
    #     "description": "Stage 1: full prediction head, frozen backbone",
    # },
    # "stage2_last_blocks": {
    #     "config": STAGE2["config"],
    #     "weights": STAGE2["ckpt_dir"] / "stage2_weights.bin",
    #     "description": "Stage 2: full prediction head + last DINO blocks",
    # },
    "stage2_last_blocks_resumed_40epochs": {
        "config": STAGE2["config"],
        "weights": STAGE2["ckpt_dir"] / "stage2_weights_rerun.bin",
        "description": "Stage 2 rerun with 40 epoches, no early stopping",
    },
}

for name, info in TASK5_MODELS.items():
    print(f"\n{name}")
    print(" description:", info["description"])
    print(" config:", info["config"], "| exists:", info["config"].exists())
    print(" weights:", info["weights"], "| exists:", info["weights"].exists())


stage2_last_blocks_resumed_40epochs
 description: Stage 2 rerun with 40 epoches, no early stopping
 config: /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject/eomt/configs/dinov2/finetuning/coco_to_cityscapes_unfreeze_last.yaml | exists: True
 weights: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/weights/finetuned/coco_to_cityscapes_stage2_unfreeze_last/stage2_weights_rerun.bin | exists: True


In [28]:
# Evaluate all fine-tuned checkpoints

task5_results = {}

for model_name, info in TASK5_MODELS.items():
    print("\n" + "=" * 80)
    print(f"Evaluating: {model_name}")
    print(info["description"])
    print("=" * 80)

    assert info["config"].exists(), f"Missing config: {info['config']}"
    assert info["weights"].exists(), f"Missing weights: {info['weights']}"

    model = build_eomt_eval_model(
        config_path=info["config"],
        weights_path=info["weights"],
        img_size=IMG_SIZE,
        num_classes=19,
        device=TASK5_EVAL_DEVICE,
        force_masked_attn_enabled=False,
    )

    per_class_iou, miou, pixel_acc = evaluate_cityscapes_semantic(
        model=model,
        dataset=eval_dataset,
        output_space="cityscapes",
        excluded_classes=TASK5_EXCLUDED_CLASSES,
        device=TASK5_EVAL_DEVICE,
        model_name=model_name,
    )

    task5_results[model_name] = {
        "description": info["description"],
        "per_class_iou": per_class_iou,
        "miou": miou,
        "pixel_acc": pixel_acc,
    }

    print(f"{model_name} | mIoU={miou:.2f}% | pixel_acc={pixel_acc:.2f}%")

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()



Evaluating: stage2_last_blocks_resumed_40epochs
Stage 2 rerun with 40 epoches, no early stopping


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


[stage2_last_blocks_resumed_40epochs] 50/500 images processed
[stage2_last_blocks_resumed_40epochs] 100/500 images processed
[stage2_last_blocks_resumed_40epochs] 150/500 images processed
[stage2_last_blocks_resumed_40epochs] 200/500 images processed
[stage2_last_blocks_resumed_40epochs] 250/500 images processed
[stage2_last_blocks_resumed_40epochs] 300/500 images processed
[stage2_last_blocks_resumed_40epochs] 350/500 images processed
[stage2_last_blocks_resumed_40epochs] 400/500 images processed
[stage2_last_blocks_resumed_40epochs] 450/500 images processed
[stage2_last_blocks_resumed_40epochs] 500/500 images processed
stage2_last_blocks_resumed_40epochs | mIoU=72.98% | pixel_acc=95.04%


In [29]:
# Build and save Task 5 evaluation tables

df_summary, df_per_class = make_iou_tables(
    results=task5_results,
    excluded_classes=TASK5_EXCLUDED_CLASSES,
)

display(df_summary)
display(df_per_class)

from datetime import datetime

out_xlsx = EVAL_RESULTS_DIR / f"task5_finetuned_cityscapes_evaluation_19class_{datetime.now().strftime('%Y-%m-%d_%H-%M')}.xlsx"

with pd.ExcelWriter(out_xlsx, engine="xlsxwriter") as writer:
    df_summary.to_excel(writer, sheet_name="summary", index=False)
    df_per_class.to_excel(writer, sheet_name="per_class_iou", index=False)

print("Saved Task 5 evaluation results to:", out_xlsx)

,model,description,mIoU (%),pixel_acc (%)
0,stage2_last_blocks_resumed_40epochs,"Stage 2 rerun with 40 epoches, no early stopping",72.980507,95.040315


,class_id,class_name,excluded,IoU stage2_last_blocks_resumed_40epochs (%)
0,0,road,False,97.612587
1,1,sidewalk,False,81.304207
2,2,building,False,91.339745
3,3,wall,False,58.320869
4,4,fence,False,51.782944
5,5,pole,False,49.539143
6,6,traffic light,False,60.898979
7,7,traffic sign,False,69.150475
8,8,vegetation,False,90.642647
9,9,terrain,False,64.082191


Saved Task 5 evaluation results to: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results/task5/task5_finetuned_cityscapes_evaluation_19class_2026-06-05_08-49.xlsx
